# EDA 01: LGD district spine and name map

**EDA step 1.** Every table in this project is keyed by the **LGD district code**: the Local
Government Directory's permanent ID for a district. This notebook builds the *spine*, the
one row per district that every other source joins to, and the *name map* that turns the
district names other sources print into LGD codes.

| | |
|---|---|
| **Input** | `data/raw/lgd/ingest_date=YYYY-MM-DD/Districts_of_Uttar_Pradesh_State.pdf` (downloaded by hand from lgdirectory.gov.in) |
| **Hand-curated** | `dbt/seeds/district_aliases.csv`, `dbt/seeds/district_lineage.csv` |
| **Outputs** | `data/interim/lgd/up_district_spine.parquet`, `dbt/seeds/lgd_district_spine.csv`, `dbt/seeds/district_name_map.csv` |
| **Run** | `make lab`, then *Run All*. The test suite also runs it (`tests/test_notebooks.py`). |

**How every section works (the notebook EDA pattern).** Each section answers **one question**,
stated in its first markdown cell. Code cells do the work. The section ends with an
**`assert` cell** that encodes the answer, so if the data changes, *Run All* stops at the first
broken assumption instead of carrying on with a wrong spine. Findings go in the last section
and in `docs/eda/`.

In [1]:
import os
import re
import unicodedata
from difflib import get_close_matches
from pathlib import Path

import duckdb
import pandas as pd

from up_siting.common.names import normalise_name
from up_siting.common.paths import INTERIM_DIR, RAW_DIR, SEEDS_DIR
from up_siting.extract.lgd import parse_lgd_districts_pdf

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

# Parameters. Environment variables let the test suite point the notebook at fixtures
# and temporary folders; normally you leave them unset.
EXPECTED_DISTRICTS = 75
LGD_STATE_CODE_UP = 9  # LGD (and Census) state code for Uttar Pradesh
OUT_DIR = Path(os.environ.get("UP_SITING_OUT_DIR", INTERIM_DIR / "lgd"))
SEED_OUT_DIR = Path(os.environ.get("UP_SITING_SEED_OUT_DIR", SEEDS_DIR))

if "UP_SITING_LGD_PDF" in os.environ:
    LGD_PDF = Path(os.environ["UP_SITING_LGD_PDF"])
else:
    # Newest download wins: folders are named ingest_date=YYYY-MM-DD, so they sort by date.
    candidates = sorted(RAW_DIR.glob("lgd/ingest_date=*/Districts_of_Uttar_Pradesh_State.pdf"))
    assert candidates, "No LGD PDF found: see docs/data-sources.md for where to save it"
    LGD_PDF = candidates[-1]

match = re.search(r"ingest_date=(\d{4}-\d{2}-\d{2})", str(LGD_PDF))
INGEST_DATE = match.group(1) if match else None
print(f"input: {LGD_PDF.name}  ingest_date: {INGEST_DATE}")

input: Districts_of_Uttar_Pradesh_State.pdf  ingest_date: 2026-11-21


## 1. What is in the raw file?

LGD exports the list as a PDF, so we extract the table first
(`src/up_siting/extract/lgd.py`, the PDF-table worked example; read its docstring for why
`extract_tables()` alone loses half the rows). The extractor keeps **every value as text**:
it reports what the file says, and cleaning happens below where we can see it.

Then a first profile with **DuckDB**, which can run SQL directly on a pandas DataFrame in
scope (here `raw`), with no loading step.

In [2]:
raw = parse_lgd_districts_pdf(LGD_PDF)
print(raw.shape)
raw.head()

(75, 11)


,s_no,lgd_district_code,district_name_en,district_name_local,hierarchy,short_name,census_2001_code,census_2011_code,pesa_status,_page,_source_file
0,1,118,Agra,AGRA,Uttar Pradesh(State),AGR,15,146,Not Covered,1,Districts_of_Uttar_Pradesh_State.pdf
1,2,119,Aligarh,ALIGARH,Uttar Pradesh(State),ALG,12,143,Not Covered,1,Districts_of_Uttar_Pradesh_State.pdf
2,3,121,Ambedkar Nagar,AMBEDKAR NAGAR,Uttar Pradesh(State),AMN,48,178,Not Covered,1,Districts_of_Uttar_Pradesh_State.pdf
3,4,640,Amethi,Amethi,Uttar Pradesh(State),AME,,000,Not Covered,1,Districts_of_Uttar_Pradesh_State.pdf
4,5,154,Amroha,AMROHA,Uttar Pradesh(State),AMR,06,137,Not Covered,1,Districts_of_Uttar_Pradesh_State.pdf


In [3]:
# One row per column: how many distinct values, how many blanks. Unpivoting with UNPIVOT
# keeps the query the same however many columns the file has.
profile = duckdb.sql("""
    with long as (
        unpivot (select * exclude (_page, _source_file) from raw)
        on columns(*) into name column_name value value
    )
    select column_name,
           count(distinct value)                       as n_distinct,
           count(*) filter (where trim(value) = '')    as n_blank,
           min(value)                                  as min_value,
           max(value)                                  as max_value
    from long
    group by column_name
    order by column_name
""").df()
profile

,column_name,n_distinct,n_blank,min_value,max_value
0,census_2001_code,71,5,,70
1,census_2011_code,72,0,000,202
2,district_name_en,75,0,Agra,Varanasi
3,district_name_local,75,0,         ,VARANASI
4,hierarchy,1,0,Uttar Pradesh(State),Uttar Pradesh(State)
5,lgd_district_code,75,0,118,661
6,pesa_status,1,0,Not Covered,Not Covered
7,s_no,75,0,1,9
8,short_name,75,0,AGR,VRN


In [4]:
assert len(raw) == EXPECTED_DISTRICTS, f"expected {EXPECTED_DISTRICTS} rows, got {len(raw)}"
assert raw["hierarchy"].eq("Uttar Pradesh(State)").all(), "a row from another state?"

## 2. Is the LGD code a clean primary key?

A primary key must be **present, unique and stable**. Stability we can't test from one
file; that's why LGD codes, not names, are the key. Presence and uniqueness we can.

In [5]:
codes = raw["lgd_district_code"]
key_check = duckdb.sql("""
    select count(*)                                              as n_rows,
           count(distinct lgd_district_code)                     as n_distinct,
           count(*) filter (where not regexp_full_match(lgd_district_code, '[0-9]+'))
                                                                 as n_not_numeric,
           min(cast(lgd_district_code as integer))               as min_code,
           max(cast(lgd_district_code as integer))               as max_code
    from raw
""").df()
key_check

,n_rows,n_distinct,n_not_numeric,min_code,max_code
0,75,75,0,118,661


In [6]:
# Most codes are 118-187 (the original allocation); the districts created later have
# codes in the 600s. Which ones?
raw.loc[codes.astype(int) > 187, ["lgd_district_code", "district_name_en", "census_2011_code"]]

,lgd_district_code,district_name_en,census_2011_code
3,640,Amethi,000
34,661,Hapur,000
43,633,Kasganj,202
64,659,Sambhal,000
67,660,Shamli,000


In [7]:
row = key_check.iloc[0]
assert row.n_rows == row.n_distinct == EXPECTED_DISTRICTS, "LGD codes are not unique"
assert row.n_not_numeric == 0, "non-numeric LGD code"
assert raw["district_name_en"].str.strip().ne("").all(), "district with no English name"

## 3. What needs cleaning before this is a spine?

Things the profile in section 1 shows:

* **`district_name_local`** is meant to be the Hindi name, but holds upper-case English for
  72 districts, title case for 2, and for **Mirzapur, 9 NUL characters** (`\x00`): the PDF
  font has no Unicode mapping for whatever was there. PostgreSQL rejects NUL in text, so
  this would break the Phase 2 load. We treat it as missing, and since the column
  adds nothing over the English name, the spine drops it.
* **Census codes**: blank (`''`) and `'000'` both mean *no code*. Codes stay **text**
  because Census 2001 codes have leading zeros (`'06'`).
* **`short_name`** is a 3-letter code that still spells old names: `FZB` (Faizabad) for
  Ayodhya, `ALB` (Allahabad) for Prayagraj, `SRD` (Sant Ravidas Nagar) for Bhadohi. Kept:
  it's useful evidence for the alias table.

In [8]:
def has_control_chars(s: str) -> bool:
    return any(unicodedata.category(ch) == "Cc" for ch in s)


raw.loc[
    raw["district_name_local"].map(has_control_chars), ["district_name_en", "district_name_local"]
]

,district_name_en,district_name_local
55,Mirzapur,         


In [9]:
def code_or_na(s: pd.Series) -> pd.Series:
    # '' and '000' are LGD's ways of saying "no census code".
    return s.str.strip().replace({"": pd.NA, "000": pd.NA})


spine = pd.DataFrame(
    {
        "lgd_district_code": raw["lgd_district_code"].astype("int64"),
        "district_name": raw["district_name_en"].str.strip(),
        "lgd_state_code": LGD_STATE_CODE_UP,
        "short_name": raw["short_name"].str.strip(),
        "census_2001_code": code_or_na(raw["census_2001_code"]),
        "census_2011_code": code_or_na(raw["census_2011_code"]),
    }
)
spine["in_census_2001"] = spine["census_2001_code"].notna()
spine["in_census_2011"] = spine["census_2011_code"].notna()
spine["_source_file"] = raw["_source_file"]
spine["_ingest_date"] = INGEST_DATE
spine = spine.sort_values("lgd_district_code", ignore_index=True)

spine.loc[~spine["in_census_2001"] | ~spine["in_census_2011"]]

,lgd_district_code,district_name,lgd_state_code,short_name,census_2001_code,census_2011_code,in_census_2001,in_census_2011,_source_file,_ingest_date
70,633,Kasganj,9,KAS,<NA>,202,False,True,Districts_of_Uttar_Pradesh_State.pdf,2026-11-21
71,640,Amethi,9,AME,<NA>,<NA>,False,False,Districts_of_Uttar_Pradesh_State.pdf,2026-11-21
72,659,Sambhal,9,SBL,<NA>,<NA>,False,False,Districts_of_Uttar_Pradesh_State.pdf,2026-11-21
73,660,Shamli,9,SML,<NA>,<NA>,False,False,Districts_of_Uttar_Pradesh_State.pdf,2026-11-21
74,661,Hapur,9,HAP,<NA>,<NA>,False,False,Districts_of_Uttar_Pradesh_State.pdf,2026-11-21


In [10]:
# Districts that a Census-2011-based source (population, agriculture) will NOT have:
not_in_2011 = set(spine.loc[~spine["in_census_2011"], "district_name"])
not_in_2001 = set(spine.loc[~spine["in_census_2001"], "district_name"])
assert not_in_2011 == {"Amethi", "Hapur", "Sambhal", "Shamli"}, not_in_2011
assert not_in_2001 == not_in_2011 | {"Kasganj"}, not_in_2001
assert not spine.map(lambda v: isinstance(v, str) and has_control_chars(v)).any().any()

## 4. Do district names stay unique after normalising?

`normalise_name` (in `src/up_siting/common/names.py`) turns a name into a join key: case,
spaces, punctuation and words like "district" are removed, so *Bara Banki*,
*BARABANKI* and *Barabanki District* all become `barabanki`. If two **different** districts
normalised to the same key, name matching could never tell them apart, so we check now.

In [11]:
spine["district_name_normalised"] = spine["district_name"].map(normalise_name)
dupes = spine[spine.duplicated("district_name_normalised", keep=False)]
dupes

,lgd_district_code,district_name,lgd_state_code,short_name,census_2001_code,census_2011_code,in_census_2001,in_census_2011,_source_file,_ingest_date,district_name_normalised


In [12]:
assert dupes.empty, "two districts share a normalised name"
assert spine["district_name_normalised"].notna().all()

## 5. Do the hand-curated aliases agree with LGD?

`dbt/seeds/district_aliases.csv` lists names that other sources use: former names
(*Allahabad*), spelling variants (*Shravasti*), alternate names (*Lakhimpur Kheri*). Each row
carries the LGD code **and** the LGD name it's meant for, so a mistyped code shows up as a
name mismatch. `evidence` says why we believe the alias: `lgd_short_name` means the LGD
short code backs it up, `general_knowledge` means it still needs a citation.

Fuzzy matching (`difflib`) appears here only to **suggest** fixes for a human, never to match
on its own. An automatic fuzzy match would quietly send *Hamirpur* (UP) data to *Hamirpur*
(Himachal) one day.

In [13]:
aliases = pd.read_csv(SEEDS_DIR / "district_aliases.csv", dtype={"lgd_district_code": "int64"})
check = aliases.merge(
    spine[["lgd_district_code", "district_name"]],
    on="lgd_district_code",
    how="left",
    validate="many_to_one",
    indicator=True,
)
problems = check[
    (check["_merge"] != "both") | (check["lgd_district_name"] != check["district_name"])
]
for _, p in problems.iterrows():
    hint = get_close_matches(p["lgd_district_name"], spine["district_name"].tolist(), n=1)
    print(
        f"code {p.lgd_district_code}: alias file says {p.lgd_district_name!r}, "
        f"LGD says {p.district_name!r}; closest LGD name: {hint}"
    )
aliases["alias_type"].value_counts()

alias_type
spelling_variant    13
former_name         12
alternate_name       7
Name: count, dtype: int64

In [14]:
assert problems.empty, "fix dbt/seeds/district_aliases.csv (see printout above)"
assert aliases["alias_type"].isin(["former_name", "spelling_variant", "alternate_name"]).all()

## 6. Build the name map: is every alias unambiguous?

The name map is a lookup table: `alias_normalised` to `lgd_district_code`. It's the union of
the LGD English names and the hand-curated aliases. The hard rule: **one key, one district**.
If a key pointed at two districts, joining a source to the map would duplicate its rows,
the classic fan-out bug.

In [15]:
official = spine.assign(alias=spine["district_name"], alias_type="lgd_official")
name_map = pd.concat(
    [
        official[["lgd_district_code", "alias", "alias_type"]],
        aliases[["lgd_district_code", "alias", "alias_type"]],
    ],
    ignore_index=True,
)
name_map["alias_normalised"] = name_map["alias"].map(normalise_name)

# Several spellings can share a key ("Bara Banki" and "Barabanki"). That's fine as long as
# they point at the same district: keep the official row first, then drop repeats.
name_map["_priority"] = name_map["alias_type"].eq("lgd_official").map({True: 0, False: 1})
name_map = (
    name_map.sort_values(["alias_normalised", "_priority"])
    .drop_duplicates(["alias_normalised", "lgd_district_code"])
    .drop(columns="_priority")
)

ambiguous = duckdb.sql("""
    select alias_normalised, list(distinct lgd_district_code) as codes
    from name_map group by alias_normalised having count(distinct lgd_district_code) > 1
""").df()
ambiguous

,alias_normalised,codes


In [16]:
name_map = name_map[["alias_normalised", "lgd_district_code", "alias", "alias_type"]]
name_map = name_map.sort_values(["lgd_district_code", "alias_type", "alias"], ignore_index=True)
print(f"{len(name_map)} aliases for {name_map['lgd_district_code'].nunique()} districts")
name_map.head(10)

107 aliases for 75 districts


,alias_normalised,lgd_district_code,alias,alias_type
0,agra,118,Agra,lgd_official
1,aligarh,119,Aligarh,lgd_official
2,allahabad,120,Allahabad,former_name
3,prayagraj,120,Prayagraj,lgd_official
4,ambedkarnagar,121,Ambedkar Nagar,lgd_official
5,auraiya,122,Auraiya,lgd_official
6,azamgarh,123,Azamgarh,lgd_official
7,baghpat,124,Baghpat,lgd_official
8,bagpat,124,Bagpat,spelling_variant
9,bahraich,125,Bahraich,lgd_official


In [17]:
assert ambiguous.empty, "an alias points at two districts"
assert name_map["alias_normalised"].is_unique
assert set(name_map["lgd_district_code"]) == set(spine["lgd_district_code"])

## 7. Which districts were carved out of others, and does LGD agree with our lineage table?

Sources older than a split report the **parent** district's figure for land that's now in
the child. The name map can't fix that, so `dbt/seeds/district_lineage.csv` records each
split (child, parent, year). The cost model will need it to apportion or flag older data.

The LGD file gives independent evidence: a district missing from Census 2001 was created
after 2001. So the set of children in the lineage table must equal the set of districts
missing a Census 2001 code.

In [18]:
lineage = pd.read_csv(SEEDS_DIR / "district_lineage.csv")
names = spine.set_index("lgd_district_code")["district_name"]
lineage["child_ok"] = lineage["child_lgd_code"].map(names).eq(lineage["child_name"])
lineage["parent_ok"] = lineage["parent_lgd_code"].map(names).eq(lineage["parent_name"])
lineage

,child_lgd_code,child_name,parent_lgd_code,parent_name,created_year,evidence,verified,note,child_ok,parent_ok
0,633,Kasganj,138,Etah,2008,lgd_census_codes,False,No Census 2001 code; Census 2011 gives Etah 20...,True,True
1,640,Amethi,185,Sultanpur,2010,lgd_census_codes,False,Census 2011 code 000; formed from parts of Sul...,True,True
2,640,Amethi,175,Rae Bareli,2010,lgd_census_codes,False,See row above,True,True
3,661,Hapur,145,Ghaziabad,2011,lgd_census_codes,False,Census 2011 code 000,True,True
4,660,Shamli,172,Muzaffarnagar,2011,lgd_census_codes,False,Census 2011 code 000,True,True
5,659,Sambhal,171,Moradabad,2011,lgd_census_codes,False,Census 2011 code 000,True,True
6,659,Sambhal,133,Budaun,2011,general_knowledge,False,Gunnaur tehsil believed to come from Budaun: c...,True,True


In [19]:
assert lineage[["child_ok", "parent_ok"]].all().all(), "lineage codes/names disagree with LGD"
assert set(lineage["child_name"]) == not_in_2001, set(lineage["child_name"]) ^ not_in_2001

## 8. Write the outputs

* `data/interim/lgd/up_district_spine.parquet` is the spine for other notebooks (git-ignored).
* `dbt/seeds/lgd_district_spine.csv` and `dbt/seeds/district_name_map.csv` are committed, so
  the spine and name map are reviewable in git and ready for dbt in Phase 3. (Whether the
  spine stays a seed or becomes a dbt model over the raw LGD table is a *Decision needed*.)

In [20]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED_OUT_DIR.mkdir(parents=True, exist_ok=True)

spine_cols = [
    "lgd_district_code",
    "district_name",
    "district_name_normalised",
    "lgd_state_code",
    "short_name",
    "census_2001_code",
    "census_2011_code",
    "in_census_2001",
    "in_census_2011",
    "_source_file",
    "_ingest_date",
]
spine = spine[spine_cols]
spine.to_parquet(OUT_DIR / "up_district_spine.parquet", index=False)
spine.to_csv(SEED_OUT_DIR / "lgd_district_spine.csv", index=False)
name_map.to_csv(SEED_OUT_DIR / "district_name_map.csv", index=False)
print("wrote", OUT_DIR / "up_district_spine.parquet")
print("wrote", SEED_OUT_DIR / "lgd_district_spine.csv")
print("wrote", SEED_OUT_DIR / "district_name_map.csv")

wrote /home/user/up-geospatial-project/data/interim/lgd/up_district_spine.parquet
wrote /home/user/up-geospatial-project/dbt/seeds/lgd_district_spine.csv
wrote /home/user/up-geospatial-project/dbt/seeds/district_name_map.csv


In [21]:
# Read back what we wrote: the file on disk, not the variable in memory, is the deliverable.
# (DuckDB 2.0: a relation re-runs on every call, so take rows with fetchall(), not fetchone().)
[back] = duckdb.sql(f"""
    select count(*) as n, count(distinct lgd_district_code) as n_codes
    from read_parquet('{OUT_DIR / "up_district_spine.parquet"}')
""").fetchall()
assert back == (EXPECTED_DISTRICTS, EXPECTED_DISTRICTS), back
nm = pd.read_csv(SEED_OUT_DIR / "district_name_map.csv")
assert nm["alias_normalised"].is_unique and nm["lgd_district_code"].nunique() == EXPECTED_DISTRICTS

## 9. Findings

From the LGD export downloaded 2026-11-21:

1. **75 districts**, LGD codes unique and numeric. 70 codes run 118–187; Kasganj (633),
   Amethi (640), Sambhal (659), Shamli (660) and Hapur (661) were allocated later.
2. **Four districts aren't in Census 2011** (Amethi, Hapur, Sambhal, Shamli) and **five aren't
   in Census 2001** (those four plus Kasganj). Any Census-based variable has gaps there:
   see `district_lineage.csv` and the data-quality log.
3. **Etah's Census 2011 code (201) is out of sequence**, as is Kasganj's (202): Census 2011
   re-coded Etah after Kasganj was split off. Don't assume Census 2001 and 2011 codes map
   one-to-one by position.
4. The **local-language column isn't Hindi**: it's upper-case English, and **Mirzapur's value
   is 9 NUL bytes**. Dropped. If a source prints Hindi names, add Devanagari aliases.
5. **Short names still use old names** (`FZB`, `ALB`, `SRD`), which corroborate three renames.
6. LGD's own spellings to remember: *Bara Banki*, *Mahrajganj*, *Shrawasti*, *Kheri* (not
   Lakhimpur Kheri), *Rae Bareli*, *Budaun*.

Logged in `docs/eda/data-quality-log.md` (DQ-001 to DQ-004); write-up in `docs/eda/01_spine.md`.